# DUK leg motion workbench

This notebook studies **motion of the physical DUK leg mechanism** using the reusable `duk_core` model.

Model commitments used here:

- robot frame: `+Ox` right, `+Oy` up, counter-clockwise positive;
- directed `D→A = -30°`, therefore `A→D = +150°`;
- physical four-bar assembly branch: `-1`;
- **B, C and O are fixed material points of one rigid BCO wheel-carrier body**;
- `H` is only an auxiliary construction point in that same rigid body;
- the notebook calls the core model; it does not reimplement loop-closure equations.

The time-domain section below is a kinematic motion demonstration, **not yet a motor/controller model**. Actuator limits, body mass/inertia, contact forces and wheel rolling dynamics will be added only when their physical data is available.


In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np

from duk_core.modeling import (
    jacobian_in_robot_frame,
    leg_jacobian,
    load_leg_geometry_json,
    pose_in_robot_frame,
    sample_leg_trajectory,
    solve_leg_pose,
)
from duk_core.visualization import draw_leg_linkage, draw_point_trajectories


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'robot/config/physical/duk_leg_geometry.json').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from a DUK repository checkout')


repo_root = find_repo_root()
geometry = load_leg_geometry_json(repo_root / 'robot/config/physical/duk_leg_geometry.json')
print(f'physical branch = {geometry.assembly_branch:+d}')
print(f'DA angle = {math.degrees(geometry.da_angle_rad):+.1f} deg')
print(f'AD angle = {math.degrees(geometry.ad_angle_rad):+.1f} deg')
geometry


## 1. Rigid BCO carrier geometry

In the BCO body frame, B is the origin and `+x` is B→C. The material-point coordinates are fixed:

- `B = (0, 0)`
- `C = (BC, 0)`
- `H = (-BH, 0)`
- `O = (-BH, wheel_side·OH)`

Therefore `BC`, `BO`, `CO` and the signed triangle area are invariants. The mechanism can translate/rotate this triangle, but it cannot deform it.


In [ ]:
carrier = geometry.bco_body
print(f'BC = {carrier.bc_mm:.3f} mm')
print(f'BO = {carrier.bo_mm:.3f} mm')
print(f'CO = {carrier.co_mm:.3f} mm')
print(f'O in BCO-local frame = {carrier.o_from_b_local_mm} mm')
print(f'signed 2x area(B,C,O) = {carrier.signed_double_area_mm2:.3f} mm^2')


## 2. Sweep the physical mechanism through its geometric workspace

The exact singular boundaries are excluded by a small margin. This sweep uses the configured physical branch `-1`.


In [ ]:
lower, upper = geometry.geometric_angle_limits_rad()
margin = math.radians(2.0)
solver_poses = sample_leg_trajectory(
    geometry,
    lower + margin,
    upper - margin,
    181,
)
robot_poses = tuple(pose_in_robot_frame(geometry, p) for p in solver_poses)
angles_deg = np.array([math.degrees(p.input_angle_rad) for p in robot_poses])
print(f'sampled {len(robot_poses)} poses from {angles_deg[0]:.2f}° to {angles_deg[-1]:.2f}°')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for index in np.linspace(0, len(robot_poses) - 1, 5, dtype=int):
    draw_leg_linkage(
        robot_poses[index],
        geometry,
        ax=ax,
        show_auxiliary=False,
        annotate=False,
    )
ax.set_title('DUK leg motion snapshots — rigid BCO carrier')
plt.show()


In [ ]:
ax = draw_point_trajectories(robot_poses, point_names=('B', 'C', 'O'))
ax.set_title('B, C and O trajectories — physical mechanism')
plt.show()


## 3. Verify that BCO never deforms

This is the numerical guardrail for the mechanical statement that B–C–O is one rigid link. The plotted values are deviations from the nominal rigid-body geometry, not the raw distances.


In [ ]:
def distance(p, q):
    return math.hypot(p[0] - q[0], p[1] - q[1])

def cross2(a, b):
    return a[0] * b[1] - a[1] * b[0]

bc = np.array([distance(p.b, p.c) for p in robot_poses])
bo = np.array([distance(p.b, p.o) for p in robot_poses])
co = np.array([distance(p.c, p.o) for p in robot_poses])
area2 = np.array([
    cross2(
        (p.c[0] - p.b[0], p.c[1] - p.b[1]),
        (p.o[0] - p.b[0], p.o[1] - p.b[1]),
    )
    for p in robot_poses
])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(angles_deg, bc - carrier.bc_mm, label='BC error')
ax.plot(angles_deg, bo - carrier.bo_mm, label='BO error')
ax.plot(angles_deg, co - carrier.co_mm, label='CO error')
ax.set_xlabel('input angle (deg)')
ax.set_ylabel('distance error (mm)')
ax.grid(True)
ax.legend()
ax.set_title('Rigid BCO distance invariants')
plt.show()

print('max |BC error| =', float(np.max(np.abs(bc - carrier.bc_mm))), 'mm')
print('max |BO error| =', float(np.max(np.abs(bo - carrier.bo_mm))), 'mm')
print('max |CO error| =', float(np.max(np.abs(co - carrier.co_mm))), 'mm')
print('max |signed-area error| =', float(np.max(np.abs(area2 - carrier.signed_double_area_mm2))), 'mm^2')


## 4. Robot-relevant kinematic outputs

With A/D fixed in the robot frame, the wheel-center position relative to A is a useful geometric output. `A-to-contact-plane height` below assumes a horizontal contact plane tangent to the bottom of the wheel; it is a geometric anchor-height quantity, **not the robot COM height**.


In [ ]:
o_x = np.array([p.o[0] for p in robot_poses])
o_y = np.array([p.o[1] for p in robot_poses])
ao = np.array([distance(p.a, p.o) for p in robot_poses])
anchor_height = np.array([
    p.a[1] - (p.o[1] - geometry.wheel_radius_mm)
    for p in robot_poses
])
carrier_angle = np.unwrap(np.array([
    math.atan2(p.c[1] - p.b[1], p.c[0] - p.b[0])
    for p in robot_poses
]))

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0, 0].plot(angles_deg, o_x)
axes[0, 0].set_title('wheel-center fore/aft O.x')
axes[0, 1].plot(angles_deg, o_y)
axes[0, 1].set_title('wheel-center vertical O.y')
axes[1, 0].plot(angles_deg, anchor_height)
axes[1, 0].set_title('A-to-horizontal-contact-plane height')
axes[1, 1].plot(angles_deg, np.degrees(carrier_angle))
axes[1, 1].set_title('rigid BCO carrier orientation')
for ax in axes.flat:
    ax.set_xlabel('input angle (deg)')
    ax.grid(True)
plt.tight_layout()
plt.show()


## 5. Differential motion and transmission

The analytic Jacobian maps input angular rate `α̇` to wheel-center velocity. Its magnitude shows how strongly a small actuator-angle change moves O at each configuration.


In [ ]:
robot_jac = tuple(
    jacobian_in_robot_frame(geometry, leg_jacobian(geometry, p))
    for p in solver_poses
)
dox = np.array([j.d_o[0] for j in robot_jac])
doy = np.array([j.d_o[1] for j in robot_jac])
gain = np.hypot(dox, doy)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(angles_deg, dox, label='dO.x/dα')
ax.plot(angles_deg, doy, label='dO.y/dα')
ax.plot(angles_deg, gain, label='||dO/dα||')
ax.set_xlabel('input angle (deg)')
ax.set_ylabel('mm/rad')
ax.grid(True)
ax.legend()
ax.set_title('Wheel-center differential motion')
plt.show()


## 6. Time-domain motion example

A smooth sinusoidal input is used only to demonstrate how the mechanism moves over time. It is deliberately kept inside the geometric workspace. This is useful for later squat/bounce/height-motion experiments, but it is not yet a controller command profile.


In [ ]:
t = np.linspace(0.0, 5.0, 301)
safe_lower = lower + math.radians(5.0)
safe_upper = upper - math.radians(5.0)
alpha_mid = 0.5 * (safe_lower + safe_upper)
alpha_amp = 0.35 * (safe_upper - safe_lower)
omega = 2.0 * math.pi * 0.4
alpha_t = alpha_mid + alpha_amp * np.sin(omega * t)
alpha_dot_t = alpha_amp * omega * np.cos(omega * t)

time_solver_poses = tuple(solve_leg_pose(geometry, float(a)) for a in alpha_t)
time_robot_poses = tuple(pose_in_robot_frame(geometry, p) for p in time_solver_poses)
time_robot_jac = tuple(
    jacobian_in_robot_frame(geometry, leg_jacobian(geometry, p))
    for p in time_solver_poses
)

time_ox = np.array([p.o[0] for p in time_robot_poses])
time_oy = np.array([p.o[1] for p in time_robot_poses])
time_vx = np.array([j.d_o[0] for j in time_robot_jac]) * alpha_dot_t
time_vy = np.array([j.d_o[1] for j in time_robot_jac]) * alpha_dot_t

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
axes[0].plot(t, np.degrees(alpha_t))
axes[0].set_ylabel('α (deg)')
axes[0].set_title('Smooth kinematic motion example')
axes[1].plot(t, time_ox, label='O.x')
axes[1].plot(t, time_oy, label='O.y')
axes[1].set_ylabel('position (mm)')
axes[1].legend()
axes[2].plot(t, time_vx, label='O.vx')
axes[2].plot(t, time_vy, label='O.vy')
axes[2].set_ylabel('velocity (mm/s)')
axes[2].set_xlabel('time (s)')
axes[2].legend()
for ax in axes:
    ax.grid(True)
plt.tight_layout()
plt.show()


## Next motion layers

This notebook now provides the geometric basis for later DUK motion studies without changing the physical model. Natural next notebooks can build on it for:

1. inverse kinematics from desired wheel-center/anchor height;
2. two-leg differential height for slope leveling;
3. coordinated height change while wheel translation occurs;
4. body-pitch/roll compensation once both legs and body frames are modeled;
5. actuator speed/torque limits and dynamic motion after motor and mass-property data are connected.
